# UART Challenge: Noisy 8E1 Capture

**Time: 45?60 minutes.** Recover the transmitted bytes from `data/uart_8e1_capture.csv`.

- Sample rate: **48,000 Hz**. Nominal baud rate: **2,400**, with up to **?2%** clock error.
- Format: **8 data bits, even parity, 1 stop bit; LSB first**. Even parity means the eight data bits plus the parity bit contain an even number of ones.
- Electrical polarity is unknown. The capture starts and ends idle; frame alignment and gaps are unknown.
- The waveform contains noise, isolated spikes, and a slowly changing voltage offset.
- Some frames have parity or stop-bit errors. Discard those frames and preserve the order of valid bytes.

Implement `decode_capture`. Return `bytes`. Do not hardcode the message or frame positions. Use the Python standard library; plotting is allowed.

In [5]:
import csv

SAMPLE_RATE = 48_000
BAUD_RATE = 2_400

with open("data/uart_8e1_capture.csv", newline="") as file:
    samples = [float(row["voltage"]) for row in csv.DictReader(file)]

print(f"Loaded {len(samples)} samples ({len(samples) / SAMPLE_RATE:.3f} seconds)")

Loaded 7837 samples (0.163 seconds)


In [ ]:
def decode_capture(samples, sample_rate, baud_rate):
    #Section 1 Filter
    SAMPLES_PER_BIT = sample_rate // baud_rate
    window = []

    #Create Sample Windows for our bits
    for i in range (0,len(samples)-SAMPLES_PER_BIT+1,SAMPLES_PER_BIT):
        window.append([abs(x) for x in samples[i:i+SAMPLES_PER_BIT]])
    ordered = []
    for x in window:
        ordered.append(sorted(x))

    bits = []

    #Median Filter
    for x in ordered:
        if (x[SAMPLES_PER_BIT // 2] >= 0.38):
            bits.append(1)
        else:
            bits.append(0)


    #Decode bytes

    j = 1 
    Decoded= []
    while j+10 < len(bits):
        falling_edge = (bits[j-1] == 1) and (bits [j] == 0)
        stop_bit = bits[j+10]
        parity = (sum(bits[j+1:j+10]) % 2) == 0
        if (falling_edge and stop_bit and parity):
            value = 0
            for k, bit in enumerate(bits[j+1:j+9]):
                value |= (bit<<k)
            Decoded.append(value)
            j+=11
        else:
            j+=1

    return bytes (Decoded)
    raise NotImplementedError

In [15]:
import hashlib

try:
    recovered = decode_capture(samples, SAMPLE_RATE, BAUD_RATE)
except NotImplementedError:
    print("Not implemented yet.")
else:
    assert isinstance(recovered, bytes), "Return bytes."
    assert hashlib.sha256(recovered).hexdigest() == "c5de746b18f6bb5fbefda21f824a881a9fa2abbee5090753b83e66f7178d5b3a", "FAIL: recovered bytes do not match."
    print("PASS")
    print(repr(recovered))

AssertionError: FAIL: recovered bytes do not match.